In [9]:
import socket
import os
from pathlib import Path
import csv
from datetime import datetime
import logging

if "__file__" in globals():
    BASE_DIR = Path(__file__).resolve().parent
else:
    BASE_DIR = Path(os.getcwd())

NUT_HOST = "172.21.0.1"
NUT_PORT = 3493
UPS_NAME = "cyberpower"

def nut_command(cmd):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(5)
        s.connect((NUT_HOST, NUT_PORT))
        s.sendall((cmd + "\n").encode())
        response = b""
        while True:
            chunk = s.recv(4096)
            if not chunk:
                break
            response += chunk
            if b"END LIST" in response or b"ERR" in response:
                break
        return response.decode()

def get_ups_vars():
    raw = nut_command(f"LIST VAR {UPS_NAME}")
    data = {}
    for line in raw.splitlines():
        if line.startswith("VAR"):
            parts = line.split(" ", 3)
            if len(parts) == 4:
                key = parts[2]
                value = parts[3].strip('"')
                data[key] = value
    return data

# 1. Obtenemos el diccionario con todos los datos
ups_data = get_ups_vars()

# 2. Extraemos las variables con un valor por defecto (0) en caso de que no existan
# Usamos int() para convertir el string que devuelve el comando a un número entero
try:
    battery_charge = int(ups_data.get('battery.charge', 0))
    ups_load = int(ups_data.get('ups.load', 0))
except (ValueError, TypeError):
    # En caso de que el dato no sea un número válido
    battery_charge = 0
    ups_load = 0

ups_watts = round(ups_load * 540 / 100, 1)

In [2]:
stats_file = BASE_DIR / "ups.csv"
stats_file.parent.mkdir(parents=True, exist_ok=True)
file_exists = stats_file.exists()

with open(stats_file, "a", newline="") as f:
    writer = csv.writer(f)
    if not file_exists:
        writer.writerow(["date", "battery_charge", "ups_watts"])
    writer.writerow([
        datetime.now().strftime("%Y-%m-%d %H:%M"),
        battery_charge,
        ups_watts
    ])

msg = f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - ups battery: {battery_charge}, instant power: {ups_watts}W"
logging.info(msg)
print(msg)

2026-04-19 19:02:00 - ups battery: 100, instant power: 91.8W


In [6]:
import pandas as pd
df = pd.read_csv(BASE_DIR / 'ups.csv')
df['date'] = pd.to_datetime(df['date'])
today = pd.Timestamp.now()
chart_start = today - pd.Timedelta(days=1)

df_filtered = df[(df['date'] >= chart_start)].copy()

ups_data = []
for _, row in df_filtered.iterrows():
    battery_charge = row['battery_charge']
    power = row['power']
    if pd.notna(battery_charge) and battery_charge > 0:
        ups_data.append({
                "date": str(row['date'].date()),
                "battery_charge": round(battery_charge, 1),
                "power": round(power, 1)
            })
ups_data.sort(key=lambda x: x['date'])

labels = [d['date'] for d in ups_data]
battery_charge = [d['battery_charge'] for d in ups_data]
power = [d['power'] for d in ups_data]

print(today, chart_start)

2026-04-12 17:12:29.402432 2026-04-11 17:12:29.402432


In [8]:
import psutil
from datetime import datetime
import subprocess

now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

#CPU
cpu_cores = psutil.cpu_percent(percpu=True, interval=1)
cpu_avg = round(sum(cpu_cores) / len(cpu_cores), 1)

# RAM
mem = psutil.virtual_memory()
mem_pct = round((mem.active / mem.total) * 100, 1)

# Tank HDD usage
used, avail = subprocess.check_output(
    ["zfs", "list", "-H", "-p", "-o", "used,avail", "tank-hdd"],
    text=True
).strip().split("\t")

used = int(used)
avail = int(avail)

disk_tank_pct = round((used / (used + avail)) * 100, 1)

row = {
    'datetime': now,
    'cpu_avg': round(cpu_avg, 1),
    'ram_pct': mem_pct,
    'disk_tank_pct': disk_tank_pct,
}

# Agregar cada core
for i, pct in enumerate(cpu_cores):
    row[f'cpu_core_{i}'] = pct

print(f"{now} - CPU (avg over 10s): {cpu_avg}% RAM: {mem_pct}% Tank: {disk_tank_pct}%")

# Retrieve all temperature sensors
temps = psutil.sensors_temperatures()

# Check if 'k10temp' is available and extract 'Tdie'
if 'k10temp' in temps:
    for entry in temps['k10temp']:
        if entry.label == 'Tdie':
            tdie_temp = entry.current
            print(tdie_temp)

2026-04-20 14:35:43 - CPU (avg over 10s): 1.2% RAM: 23.0% Tank: 73.8%
32.25


In [20]:
import pandas as pd

df = pd.read_csv(BASE_DIR / 'metrics.csv')
df['date'] = pd.to_datetime(df['date'])
today = pd.Timestamp.now()
chart_start = today - pd.Timedelta(days=1)

df_filtered = df[(df['date'] >= chart_start)].copy()

cpu_data = []
for _, row in df_filtered.iterrows():
    cpu_avg_trend = row['cpu_avg_10s']

    tdie_trend = row['Tdie']
    if pd.notna(tdie_trend) and tdie_trend > 0:
        cpu_data.append({
                "date": str(row['date'].strftime('%d %H:%M')),
                "cpu_usage": cpu_avg_trend,
                "Tdie": tdie_trend
            })
# Function to calculate moving average of a list over a window size
def moving_average(data, window_size):
    if len(data) < window_size:
        return None
    return sum(data[-window_size:]) / window_size

# Add the moving average for cpu_usage
cpu_data_with_ma = []
for i in range(len(cpu_data)):
    ma_value = moving_average([item['cpu_usage'] for item in cpu_data[:i+1]], 3)
    if ma_value is not None:
        cpu_data_with_ma.append({
            "date": cpu_data[i]['date'],
            "cpu_usage": cpu_data[i]['cpu_usage'],
            "Tdie": cpu_data[i]['Tdie'],
            "cpu_usage_3_avg": round(ma_value, 1)
        })

# Sort by date to ensure the correct order
cpu_data_with_ma.sort(key=lambda x: x['date'])

print(cpu_data_with_ma)

[{'date': '20 14:44', 'cpu_usage': 6.2, 'Tdie': 32.6, 'cpu_usage_3_avg': 2.8}, {'date': '20 14:46', 'cpu_usage': 0.8, 'Tdie': 31.8, 'cpu_usage_3_avg': 2.8}, {'date': '20 14:48', 'cpu_usage': 0.7, 'Tdie': 31.5, 'cpu_usage_3_avg': 2.6}, {'date': '20 14:50', 'cpu_usage': 1.6, 'Tdie': 31.5, 'cpu_usage_3_avg': 1.0}, {'date': '20 14:52', 'cpu_usage': 0.8, 'Tdie': 31.8, 'cpu_usage_3_avg': 1.0}, {'date': '20 14:54', 'cpu_usage': 2.2, 'Tdie': 32.6, 'cpu_usage_3_avg': 1.5}, {'date': '20 14:56', 'cpu_usage': 0.7, 'Tdie': 32.0, 'cpu_usage_3_avg': 1.2}, {'date': '20 14:58', 'cpu_usage': 1.0, 'Tdie': 31.8, 'cpu_usage_3_avg': 1.3}, {'date': '20 15:00', 'cpu_usage': 2.0, 'Tdie': 31.5, 'cpu_usage_3_avg': 1.2}, {'date': '20 15:02', 'cpu_usage': 0.7, 'Tdie': 31.5, 'cpu_usage_3_avg': 1.2}, {'date': '20 15:04', 'cpu_usage': 1.0, 'Tdie': 31.8, 'cpu_usage_3_avg': 1.2}, {'date': '20 15:05', 'cpu_usage': 7.6, 'Tdie': 33.5, 'cpu_usage_3_avg': 3.1}, {'date': '20 15:06', 'cpu_usage': 1.6, 'Tdie': 32.0, 'cpu_usage

In [40]:
import pandas as pd

df = pd.read_csv(BASE_DIR / 'metrics.csv')
df['date'] = pd.to_datetime(df['date'])

chart_start = pd.Timestamp.now() - pd.Timedelta(days=1)

cpu_data = (
    df.loc[
        (df['date'] >= chart_start) &
        (df['Tdie'].notna()) &
        (df['Tdie'] > 0),
        ['date', 'cpu_avg_10s', 'Tdie']
    ]
    .copy()
    .sort_values('date')
)

cpu_data['cpu_usage_3_avg'] = (
    cpu_data['cpu_avg_10s']
    .rolling(window=3)
    .mean()
    .round(1)
)

cpu_data['cpu_usage_3_max'] = (
    cpu_data['cpu_avg_10s']
    .rolling(window=3)
    .max()
    .round(2)
)

# cpu_data = cpu_data.dropna(subset=['cpu_usage_3_avg'])

print(cpu_data.head(50))

cpu_data2 = []
for _, row in df.iterrows():
    cpu_avg_trend = row['cpu_avg_10s']
    tdie_trend = row['Tdie']
    if pd.notna(tdie_trend) and tdie_trend > 0:
        cpu_data2.append({
                "date": str(row['date'].strftime('%d %H:%M')),
                "cpu_usage": cpu_avg_trend,
            })

# Convert the list of dictionaries to a DataFrame for easier manipulation
cpu_df = pd.DataFrame(cpu_data2)
max_cpu_usage = cpu_df['cpu_usage'].max()

print(cpu_data2)
print(max_cpu_usage)

                    date  cpu_avg_10s  Tdie  cpu_usage_3_avg  cpu_usage_3_max
2930 2026-04-20 14:40:00          1.0  31.2              NaN              NaN
2931 2026-04-20 14:42:00          1.3  32.2              NaN              NaN
2932 2026-04-20 14:44:00          6.2  32.6              2.8              6.2
2933 2026-04-20 14:46:00          0.8  31.8              2.8              6.2
2934 2026-04-20 14:48:00          0.7  31.5              2.6              6.2
2935 2026-04-20 14:50:00          1.6  31.5              1.0              1.6
2936 2026-04-20 14:52:00          0.8  31.8              1.0              1.6
2937 2026-04-20 14:54:00          2.2  32.6              1.5              2.2
2938 2026-04-20 14:56:00          0.7  32.0              1.2              2.2
2939 2026-04-20 14:58:00          1.0  31.8              1.3              2.2
2940 2026-04-20 15:00:00          2.0  31.5              1.2              2.0
2941 2026-04-20 15:02:00          0.7  31.5              1.2    